# Grokking Modular Addition — Walkthrough

This notebook reproduces the key result from the project report: a one-layer transformer trained on modular addition mod 113 exhibits grokking, and the generalizing solution is a Fourier-multiplication circuit whose key frequencies are visible in the token embedding.

Run `python -m src.train_modular --steps 35000` first to produce `results/grok_p113.pt`.

In [ ]:
import json
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch
import matplotlib.pyplot as plt
from src.model import Config, TinyTransformer
from src.fourier import embedding_fourier_power, top_frequencies

## 1. The phase transition

Train accuracy saturates inside 200 steps. Test accuracy stays at chance for ~9000 steps — this is the pure-memorization plateau. Then it phase-transitions to 1.0 between steps 13k and 17k.

In [ ]:
hist = json.loads(Path('../results/grok_p113.json').read_text())['history']
steps = [r['step'] for r in hist]
plt.figure(figsize=(8, 4))
plt.plot(steps, [r['train'] for r in hist], label='train', lw=2)
plt.plot(steps, [r['test'] for r in hist], label='test', lw=2)
plt.xlabel('step'); plt.ylabel('accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.title('Grokking on (a + b) mod 113')
plt.show()

## 2. Fourier decomposition of the token embedding

After grokking, the embedding matrix `W_E ∈ R^{113×128}` should look like a few pure Fourier basis vectors rather than a random matrix. Project onto the real Fourier basis of Z/113Z and look at the spectrum.

In [ ]:
p = 113
model = TinyTransformer(Config(vocab_size=p + 1))
model.load_state_dict(torch.load('../results/grok_p113.pt', map_location='cpu'))

power = embedding_fourier_power(model.tok_embed.weight, p)

plt.figure(figsize=(10, 4))
plt.bar(range(p), power.numpy())
plt.xlabel('Fourier component index'); plt.ylabel('power')
plt.title(f'Token embedding Fourier spectrum (top 12 = {100*power.topk(12).values.sum()/power.sum():.1f}% of power)')
plt.grid(alpha=0.3, axis='y')
plt.show()

for idx, pw in top_frequencies(power, k=10):
    k = (idx + 1) // 2
    kind = 'DC' if idx == 0 else ('cos' if idx % 2 == 1 else 'sin')
    print(f'idx {idx:3d}  k={k:3d} {kind:3s}  power {pw:.2f}  ({pw/power.sum():6.1%})')

## 3. Reading the result

You should see mass concentrated on exactly five frequencies, each appearing as a cos/sin pair. Those frequencies are seed-dependent (a different random init will pick a different handful), but the *count* and the pairing structure are the load-bearing claims. The network embeds `a` as `[cos(2πk·a/p), sin(2πk·a/p)]` across its chosen frequencies, uses trig identities in the MLP to compute `(a+b) mod p`, and reads out with an inverse DFT in the unembedding.